In [4]:
# 📦 Setup and Imports
import sys, json, pathlib
from pprint import pprint

BASE = pathlib.Path.cwd()
DATA = BASE / "data"
ARTIFACTS = BASE / "artifacts"
DATA.mkdir(exist_ok=True)
ARTIFACTS.mkdir(exist_ok=True)

def load_json_safe(path):
    try:
        return json.loads(path.read_text())
    except Exception as e:
        print(f"[warn] Could not load {path.name}: {e}")
        return {}

# 📂 Load Your Data Files
legacy_map = load_json_safe(DATA / "ncbi_protein_legacy_map.json")
genotype = load_json_safe(DATA / "Genotype_variant.json")
lung_fn = load_json_safe(DATA / "Lung_Function_stuff.json")
nutrition = load_json_safe(DATA / "Nutrition_BMI.json")
infections = load_json_safe(DATA / "Psuedomonas_UK.json")
survival = load_json_safe(DATA / "Survival_pred_UK.json")
treatments = load_json_safe(DATA / "Treatment_Burden_Antibiotics.json")
comorb = load_json_safe(DATA / "Comorbidities_Complications.json")

print("✅ Loaded legacy map:", len(legacy_map))
print("✅ Genotype keys:", list(genotype.get("genotypes", {}).keys()))

# 🧬 Seed Variant Classification
seed_classification = {
    "F508del": "pathogenic",
    "G542X": "pathogenic",
    "621+1G->T": "pathogenic",
    "N1303K": "pathogenic",
    "G551D": "responsive_ivacaftor",
    "R117H": "responsive_ivacaftor"
}

eti_list = set(genotype["genotypes"].get("modulator_eligibility_lists", {}).get("FDA_ETI_responsive", []))
ivacaftor_list = set(genotype["genotypes"].get("modulator_eligibility_lists", {}).get("Ivacaftor_responsive", []))

# 🛠️ Feature Functions
def standardize_variant(v):
    return legacy_map.get(v, v)

def get_pathogenicity(v):
    v = standardize_variant(v)
    if v in seed_classification:
        return seed_classification[v]
    if v in ivacaftor_list:
        return "responsive_ivacaftor"
    if v in eti_list:
        return "responsive_ETI"
    return "uncertain"

def pathogenic_count(v1, v2):
    def is_path(c): return c in ["pathogenic", "responsive_ivacaftor", "responsive_ETI"]
    return sum(is_path(get_pathogenicity(v)) for v in [v1, v2])

def get_modulator_group(v1, v2):
    v1s, v2s = standardize_variant(v1), standardize_variant(v2)
    vs = {v1s, v2s}
    if "F508del" in vs and len(vs) == 1: return 1
    if "F508del" in vs: return 2
    if v1s in eti_list or v2s in eti_list: return 3
    if v1s in ivacaftor_list or v2s in ivacaftor_list: return 4
    return 5

def featurize(v1, v2, age=None, sex=None):
    return {
        "variant1": standardize_variant(v1),
        "variant2": standardize_variant(v2),
        "pathogenic_count": pathogenic_count(v1, v2),
        "mod_group": get_modulator_group(v1, v2),
        "age": age if age is not None else -1,
        "sex_male": 1 if str(sex).upper().startswith("M") else 0
    }

# 🧪 Sanity Check
examples = [
    ("F508del", "F508del", 15, "F"),
    ("F508del", "G542X", 12, "M"),
    ("G551D", "R117H", 20, "F"),
    ("R117H", "unknown", 10, "M"),
]
for v1, v2, age, sex in examples:
    print(v1, v2, featurize(v1, v2, age, sex))

# 💾 Save Artifacts
(ARTIFACTS / "seed_classification.json").write_text(json.dumps(seed_classification, indent=2))
(ARTIFACTS / "eti_list.json").write_text(json.dumps(sorted(list(eti_list)), indent=2))
(ARTIFACTS / "ivacaftor_list.json").write_text(json.dumps(sorted(list(ivacaftor_list)), indent=2))
print("✅ Saved feature utilities to:", ARTIFACTS)


✅ Loaded legacy map: 710
✅ Genotype keys: ['genotyping_coverage', 'f508del_status', 'most_common_variant_combinations', 'variant_prevalence', 'variant_prevalence_by_nation', 'modulator_eligibility_groups', 'modulator_eligibility_lists']
F508del F508del {'variant1': 'F508del', 'variant2': 'F508del', 'pathogenic_count': 2, 'mod_group': 1, 'age': 15, 'sex_male': 0}
F508del G542X {'variant1': 'F508del', 'variant2': 'G542X', 'pathogenic_count': 2, 'mod_group': 2, 'age': 12, 'sex_male': 1}
G551D R117H {'variant1': 'G551D', 'variant2': 'R117H', 'pathogenic_count': 2, 'mod_group': 3, 'age': 20, 'sex_male': 0}
R117H unknown {'variant1': 'R117H', 'variant2': 'unknown', 'pathogenic_count': 1, 'mod_group': 3, 'age': 10, 'sex_male': 1}
✅ Saved feature utilities to: /Users/adithyavikram/Documents/Cystic_Fibrosis_Model/artifacts
